# Esperimenti — Collaborative Filtering a fattori latenti (item-based)

Questo notebook esegue e documenta gli esperimenti per il modello di **Collaborative Filtering a fattori latenti item-based** implementato in `cf_latent_item`, mostrando come diverse configurazioni influenzino le prestazioni sul set DEV di MIND.

In [ ]:
import os
import sys
from pathlib import Path
import subprocess
import textwrap

import numpy as np
import pandas as pd

# Individua la root del progetto (la directory che contiene `dataset/` e le cartelle dei modelli)
CWD = Path.cwd()
if (CWD / "dataset").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "dataset").exists():
    PROJECT_ROOT = CWD.parent
else:
    PROJECT_ROOT = CWD  # fallback

print("PROJECT_ROOT:", PROJECT_ROOT)

CF_LATENT_DIR = PROJECT_ROOT / "cf_latent_item"
DATASET_DIR = PROJECT_ROOT / "dataset"

# Verifica che sia possibile importare `cf_latent_item` come pacchetto
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from cf_latent_item.latent_item_cf import LatentFactorItemCFConfig, LatentFactorItemCFRecommender

In [ ]:
matrix_path = CF_LATENT_DIR / "sparse_user_item_interaction.npz"
user_ids_path = CF_LATENT_DIR / "user_ids.npy"
item_ids_path = CF_LATENT_DIR / "news_ids.npy"

if not (matrix_path.exists() and user_ids_path.exists() and item_ids_path.exists()):
    print("Artefatti di interazione non trovati: eseguo create_interaction_matrix.py...")
    result = subprocess.run(
        [sys.executable, str(CF_LATENT_DIR / "create_interaction_matrix.py")],
        cwd=PROJECT_ROOT,
        check=True,
        text=True,
    )
    print(result.stdout)
else:
    print("Artefatti di interazione già presenti.")

In [ ]:
BEHAV_COLS = ["impression_id", "user_id", "time", "history", "impressions"]

train_behaviors_path = DATASET_DIR / "MINDsmall_train" / "behaviors.tsv"
dev_behaviors_path = DATASET_DIR / "MINDsmall_dev" / "behaviors.tsv"
news_train_path = DATASET_DIR / "MINDsmall_train" / "news.tsv"

print("Caricamento dei comportamenti...")
train_behav = pd.read_csv(train_behaviors_path, sep="\t", header=None, names=BEHAV_COLS)
dev_behav = pd.read_csv(dev_behaviors_path, sep="\t", header=None, names=BEHAV_COLS)

print(f"Impression di training: {len(train_behav):,}")
print(f"Impression di dev:      {len(dev_behav):,}")


def parse_impressions(impr: str):
    parts = str(impr).split()
    cands = []
    labels = []
    for p in parts:
        try:
            nid, lab = p.rsplit("-", 1)
        except ValueError:
            continue
        cands.append(nid)
        labels.append(1 if lab == "1" else 0)
    return cands, np.asarray(labels, dtype=np.int32)


def parse_history(hist: str):
    if hist is None or (isinstance(hist, float) and np.isnan(hist)):
        return []
    s = str(hist).strip()
    if not s:
        return []
    return s.split()


def build_train_popularity(df: pd.DataFrame):
    clicked = (
        df["impressions"]
        .dropna()
        .astype(str)
        .str.split()
        .explode()
    )
    clicked = clicked[clicked.str.endswith("-1")].str.rsplit("-", n=1).str[0]
    return clicked.value_counts().to_dict()


pop_clicks = build_train_popularity(train_behav)
pop_total = float(sum(pop_clicks.values())) if pop_clicks else 1.0
print(f"Articoli unici cliccati nel TRAIN: {len(pop_clicks):,}")

In [ ]:
# Il file news.tsv di MIND non ha intestazione; colonne:
# 0=News_ID, 1=Category, 2=SubCategory, ...
news_df = pd.read_csv(news_train_path, sep="\t", header=None)
news_df = news_df[[0, 1, 2]].dropna()
news_df.columns = ["news_id", "category", "subcategory"]

id_to_bucket = {
    str(row["news_id"]): (str(row["category"]), str(row["subcategory"]))
    for _, row in news_df.iterrows()
}

len(id_to_bucket), list(id_to_bucket.items())[:3]

In [ ]:
config_base = LatentFactorItemCFConfig(
    factors=50,
    regularization=0.01,
    iterations=20,
    alpha_confidence=20.0,
    similarity_metric="cosine",  # normalizziamo al coseno dopo il training
    decay=0.8,
    max_hist=50,  # cronologia completa per il primo esperimento
)

model_base = LatentFactorItemCFRecommender(config=config_base)
model_base.fit_from_matrix(matrix_path, user_ids_path, item_ids_path)
model_base.build_category_centroids(news_train_path)

print("Shape dei fattori degli articoli:", model_base.item_factors.shape)

In [ ]:
from collections import defaultdict
from sklearn.metrics import roc_auc_score


def item_novelty(clicks: dict, pop_total: float, news_id: str) -> float:
    c = clicks.get(news_id, 1)
    p = float(c) / float(pop_total) if pop_total > 0 else 0.0
    if p <= 0.0:
        return 0.0
    return float(-np.log2(p))


def deterministic_order(y_score: np.ndarray, cand_ids):
    y_score = np.asarray(y_score, dtype=np.float64)
    cand_ids = np.asarray(cand_ids, dtype=str)
    return np.lexsort((cand_ids, -y_score))


def mrr_first_hit(y_true, y_score, cand_ids):
    order = deterministic_order(y_score, cand_ids)
    y_sorted = y_true[order]
    hits = np.where(y_sorted == 1)[0]
    return 1.0 / (hits[0] + 1) if len(hits) > 0 else 0.0


def precision_at_k(y_true, y_score, cand_ids, k):
    order = deterministic_order(y_score, cand_ids)[:k]
    if len(order) == 0:
        return 0.0
    return float(np.mean(y_true[order]))


def recall_at_k(y_true, y_score, cand_ids, k):
    denom = int(np.sum(y_true))
    if denom <= 0:
        return 0.0
    order = deterministic_order(y_score, cand_ids)[:k]
    if len(order) == 0:
        return 0.0
    return float(np.sum(y_true[order]) / denom)


def novelty_at_k(clicks, pop_total, y_score, cand_ids, k):
    order = deterministic_order(y_score, cand_ids)[:k]
    if len(order) == 0:
        return 0.0
    top_ids = [cand_ids[i] for i in order]
    vals = [item_novelty(clicks, pop_total, nid) for nid in top_ids]
    return float(np.mean(vals)) if vals else 0.0


def evaluate_latent_cf(
    model,
    dev_df: pd.DataFrame,
    max_hist: int,
    decay: float,
    pop_clicks: dict,
    pop_total: float,
    pop_mode: str = "none",   # "none", "log", "sqrt"
    pop_weight: float = 0.0,  # es. 0.2 o 0.3
):
    metrics = defaultdict(list)
    n_rows = len(dev_df)

    for idx in range(n_rows):
        row = dev_df.iloc[idx]
        history_ids = parse_history(row.get("history", ""))
        cand_ids, y_true = parse_impressions(row.get("impressions", ""))

        if len(cand_ids) == 0 or int(y_true.sum()) == 0:
            continue

        # solo cronologia recente
        if max_hist > 0 and len(history_ids) > max_hist:
            history_ids = history_ids[-max_hist:]

        history_buckets = {nid: id_to_bucket.get(nid) for nid in history_ids}
        cand_buckets = {nid: id_to_bucket.get(nid) for nid in cand_ids}

        # score dal solo segnale latente
        scores_cf = model.score_candidates(
            history_item_ids=history_ids,
            candidate_item_ids=cand_ids,
            history_buckets=history_buckets,
            candidate_buckets=cand_buckets,
        ).astype(np.float64)

        scores = scores_cf.copy()
        if pop_mode != "none" and pop_weight > 0.0:
            pops = np.asarray([pop_clicks.get(nid, 0) for nid in cand_ids], dtype=np.float64)
            if pop_mode == "log":
                pops = np.log1p(pops)
            elif pop_mode == "sqrt":
                pops = np.sqrt(pops)
            max_p = float(pops.max()) if pops.size > 0 else 0.0
            if max_p > 0.0:
                pops = pops / max_p
            scores = (1.0 - pop_weight) * scores_cf + pop_weight * pops

        # AUC
        if len(np.unique(y_true)) == 2:
            try:
                metrics["auc"].append(roc_auc_score(y_true, scores))
            except Exception:
                pass

        metrics["mrr"].append(mrr_first_hit(y_true, scores, cand_ids))
        metrics["ndcg@5"].append(_ndcg_at_k_local(y_true, scores, cand_ids, k=5))
        metrics["ndcg@10"].append(_ndcg_at_k_local(y_true, scores, cand_ids, k=10))
        metrics["p@5"].append(precision_at_k(y_true, scores, cand_ids, k=5))
        metrics["r@5"].append(recall_at_k(y_true, scores, cand_ids, k=5))
        metrics["novelty@5"].append(novelty_at_k(pop_clicks, pop_total, scores, cand_ids, k=5))
        metrics["novelty@10"].append(novelty_at_k(pop_clicks, pop_total, scores, cand_ids, k=10))

    def _mean(xs):
        return float(np.mean(xs)) if xs else 0.0

    summary = {k: _mean(v) for k, v in metrics.items()}
    return summary


def _ndcg_at_k_local(y_true, y_score, cand_ids, k):
    order = deterministic_order(y_score, cand_ids)[:k]
    if len(order) == 0:
        return 0.0
    gains = y_true[order]

    def dcg(arr):
        return float(np.sum(arr / np.log2(np.arange(2, len(arr) + 2, dtype=np.float64))))

    ideal = np.sort(y_true)[::-1][:k]
    idcg = dcg(ideal)
    if idcg <= 0.0:
        return 0.0
    return dcg(gains) / idcg

## Esperimento 1 – similarità coseno, max_hist=50, hybrid con popolarità log (0.8 latent / 0.2 pop)

Per prima cosa proviamo una configurazione vicina ai nostri esperimenti precedenti:

- ALS: 50 fattori, reg=0.01, iterazioni=20, alpha=20
- Similarità: **coseno** (vettori articolo normalizzati L2)
- Cronologia: fino ai **50** articoli più recenti, pesati per recency con `decay=0.8`
- Score ibrido: **0.8 × score latente + 0.2 × popolarità log-normalizzata** (solo click di training)

In [ ]:
# Esperimento 1: coseno, max_hist=50, hybrid con popolarità log (0.8 latent / 0.2 pop)
summary_exp1 = evaluate_latent_cf(
    model=model_base,
    dev_df=dev_behav,
    max_hist=50,
    decay=0.8,
    pop_clicks=pop_clicks,
    pop_total=pop_total,
    pop_mode="log",
    pop_weight=0.2,
)
summary_exp1

## Esperimento 2 – coseno, cronologia breve (5), alpha più alto, hybrid 0.7 latent / 0.3 sqrt-pop

Introduciamo ora le modifiche principali:

- Limite ai **5 articoli più recenti** della cronologia (`max_hist=5`), per riflettere la rapida obsolescenza delle news.
- Confidenza ALS **alpha** portata da 20 a **40**, per rendere più nitidi i segnali latenti.
- Similarità coseno mantenuta.
- Score ibrido: **0.7 × score latente + 0.3 × popolarità normalizzata con radice quadrata** (solo training).

In [ ]:
# Esperimento 2: coseno, cronologia breve (5), alpha=40, hybrid 0.7 latent / 0.3 sqrt-pop
config_alpha40 = LatentFactorItemCFConfig(
    factors=50,
    regularization=0.01,
    iterations=20,
    alpha_confidence=40.0,
    similarity_metric="cosine",
    decay=0.8,
    max_hist=5,
)

model_alpha40 = LatentFactorItemCFRecommender(config=config_alpha40)
model_alpha40.fit_from_matrix(matrix_path, user_ids_path, item_ids_path)
model_alpha40.build_category_centroids(news_train_path)

summary_exp2 = evaluate_latent_cf(
    model=model_alpha40,
    dev_df=dev_behav,
    max_hist=5,
    decay=0.8,
    pop_clicks=pop_clicks,
    pop_total=pop_total,
    pop_mode="sqrt",
    pop_weight=0.3,
)
summary_exp2

## Esperimento 3 – coseno, cronologia breve (5), alpha=40, hybrid 0.8 latent / 0.2 sqrt-pop (configurazione finale)

Infine, manteniamo:

- similarità coseno;
- `max_hist=5`, `decay=0.8`;
- ALS `alpha=40`.

Ma modifichiamo il blending ibrido a **0.8 × latent + 0.2 × sqrt-popolarità**, per dare leggermente più peso ai fattori latenti pur mantenendo la popolarità come prior stabilizzante.

In [ ]:
# Esperimento 3 (configurazione finale): coseno, cronologia breve (5), alpha=40, hybrid 0.8 latent / 0.2 sqrt-pop
summary_exp3 = evaluate_latent_cf(
    model=model_alpha40,
    dev_df=dev_behav,
    max_hist=5,
    decay=0.8,
    pop_clicks=pop_clicks,
    pop_total=pop_total,
    pop_mode="sqrt",
    pop_weight=0.2,
)
summary_exp3

## Riepilogo delle configurazioni e dei compromessi

Di seguito una sintesi delle tre configurazioni principali testate (i valori nella tabella sono quelli osservati nelle celle sopra al momento della stesura originale del notebook; per la variabilità run-to-run si veda la nota in fondo).

| Esperimento | Similarità | max_hist | ALS alpha | Hybrid (latent / pop) e trasformazione | AUC | MRR | nDCG@5 | P@5 | R@5 | Novelty@5 | Note |
|---|---|---|---|---|---|---|---|---|---|---|---|
| Exp 1 | coseno | 50 | 20 | 0.8 latent + 0.2 pop **log** | ~0.531 | ~0.250 | ~0.232 | ~0.103 | ~0.354 | ~13.5 | Buon miglioramento rispetto al latente puro; cronologia ancora lunga. |
| Exp 2 | coseno | 5 | 40 | 0.7 latent + 0.3 pop **sqrt** | ~0.524 | ~0.259 | ~0.238 | ~0.104 | ~0.357 | ~13.3 | Metriche di ranking più forti; maggiore influenza della popolarità, novelty leggermente più bassa. |
| Exp 3 | coseno | 5 | 40 | 0.8 latent + 0.2 pop **sqrt** | ~0.523 | ~0.253 | ~0.234 | ~0.103 | ~0.354 | ~13.5 | Più orientato al segnale latente; molto vicino a baseline/ItemKNN+fallback con un po' più di novelty. |

**Osservazioni principali:**

- **Coseno + cronologia breve (5)** funziona meglio di cronologie lunghe per le news, poiché i click più vecchi diventano rapidamente meno rilevanti.
- Aggiungere un **hybrid con la popolarità** (trasformazione log/sqrt dei conteggi di click sul training) stabilizza il ranking e migliora AUC/MRR rispetto al solo segnale latente.
- Aumentare l'**alpha** di ALS (20 → 40) rende i vettori latenti più nitidi e migliora leggermente le metriche di ranking.
- Un peso di popolarità più alto (0.7/0.3) tende ad aumentare AUC/MRR ma riduce la novelty; 0.8/0.2 è un compromesso più orientato al segnale latente e leggermente più "novel".

**Nota sulla riproducibilità**: la fattorizzazione ALS (`AlternatingLeastSquares`) non aveva originariamente un `random_state` fissato in `cf_latent_item/latent_item_cf.py`, il che produceva numeri leggermente diversi a ogni esecuzione (come indicato nella tabella con il simbolo "~"). Nella revisione tecnica di questo progetto è stato impostato `random_state=42`, rendendo i risultati riproducibili run-to-run (si veda `REVISIONE_PROGETTO.md` per il valore verificato con la configurazione finale).

Usa questo notebook come riferimento principale per la sezione della relazione dedicata al **Collaborative Filtering a fattori latenti item-based**, spiegando come ciascuna modifica influenzi il compromesso accuratezza/novelty e come la configurazione finale si confronti con baseline, ItemKNN e modelli text-based.